# Train, Validate and Save a Keras Model on a Custom Dataset

This notebook walks through the full workflow:

1. Load and explore the custom dataset with **pandas**
2. Define a **Sequential** model with appropriate layers
3. Compile with `categorical_crossentropy` and an optimizer
4. Train with a **validation split** and **callbacks**
5. Save the trained model to an **`.h5`** file

The sample dataset (`data/customer_segments.csv`) predicts a customer **segment** (`budget`, `standard`, `premium`) from age, income, spend, visits and region.
To use your own data, change `DATA_PATH` and `TARGET` in the first code cell of Step 1.

## 0. Setup

In [ ]:
import os
import random

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

## Step 1: Load and explore the dataset with pandas

In [ ]:
DATA_PATH = "data/customer_segments.csv"   # <- change to your own CSV
TARGET = "segment"                         # <- name of the column to predict

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
# Missing values and class balance
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print()
print("Class distribution:")
print(df[TARGET].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df[TARGET].value_counts().plot.bar(ax=axes[0], color="#6c5ce7")
axes[0].set_title("Class distribution")
axes[0].set_xlabel("")

df.boxplot(column="monthly_spend", by=TARGET, ax=axes[1])
axes[1].set_title("Monthly spend by class")
axes[1].set_xlabel("")
plt.sca(axes[1]); plt.suptitle("")

df.boxplot(column="annual_income", by=TARGET, ax=axes[2])
axes[2].set_title("Annual income by class")
axes[2].set_xlabel("")
plt.sca(axes[2]); plt.suptitle("")

plt.tight_layout()
plt.show()

### Preprocessing

- Fill missing numeric values with the column median
- One-hot encode categorical features
- Encode the target to integers, then to one-hot vectors (required by `categorical_crossentropy`)
- Hold out a **test set** (20%) that the model never sees during training
- Scale features using statistics from the training data only (avoids leakage)

In [ ]:
# 1) Clean
data = df.copy()
num_cols = data.drop(columns=[TARGET]).select_dtypes(include=np.number).columns
data[num_cols] = data[num_cols].fillna(data[num_cols].median())
data = data.dropna()

# 2) Features / target
X = pd.get_dummies(data.drop(columns=[TARGET]), dtype=float)
feature_names = list(X.columns)

label_encoder = LabelEncoder()
y_int = label_encoder.fit_transform(data[TARGET])
class_names = list(label_encoder.classes_)
num_classes = len(class_names)
y = keras.utils.to_categorical(y_int, num_classes=num_classes)

# 3) Split (stratified so every split has the same class mix)
X_train, X_test, y_train, y_test, y_train_int, y_test_int = train_test_split(
    X.to_numpy(), y, y_int, test_size=0.2, stratify=y_int, random_state=SEED
)

# 4) Scale
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

print("Features   :", feature_names)
print("Classes    :", class_names)
print("Train/Test :", X_train.shape, X_test.shape)

## Step 2: Define a Sequential model

In [ ]:
model = keras.Sequential(
    [
        keras.Input(shape=(X_train.shape[1],)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation="softmax"),   # one probability per class
    ],
    name="segment_classifier",
)
model.summary()

## Step 3: Compile with `categorical_crossentropy` and an optimizer

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

## Step 4: Train with a validation split and callbacks

- `validation_split=0.2` keeps 20% of the training data for validation each epoch
- **EarlyStopping** stops when validation loss stops improving and restores the best weights
- **ReduceLROnPlateau** lowers the learning rate when progress stalls
- **CSVLogger** records every epoch to `logs/training_log.csv`

In [ ]:
os.makedirs("logs", exist_ok=True)
os.makedirs("models", exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=15, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1
    ),
    keras.callbacks.CSVLogger("logs/training_log.csv"),
]

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=32,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
hist = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hist[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
hist[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Evaluate on the held-out test set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print()
print(classification_report(y_test_int, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_test_int, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Purples")
ax.set_xticks(range(num_classes), class_names)
ax.set_yticks(range(num_classes), class_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion matrix")
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im)
plt.show()

## Step 5: Save the trained model to an `.h5` file

In [ ]:
MODEL_PATH = "models/segment_classifier.h5"
model.save(MODEL_PATH)

# The model alone is not enough at inference time: keep the scaler,
# feature order and class names too.
joblib.dump(
    {"scaler": scaler, "feature_names": feature_names, "class_names": class_names},
    "models/preprocessing.joblib",
)
print("Saved:", MODEL_PATH, "and models/preprocessing.joblib")

### Verify: reload the `.h5` file and predict

In [ ]:
loaded = keras.models.load_model(MODEL_PATH)
np.testing.assert_allclose(
    loaded.predict(X_test, verbose=0), model.predict(X_test, verbose=0), rtol=1e-5, atol=1e-6
)
print("Reloaded model reproduces the original predictions.")

In [ ]:
# Predict a brand-new customer
prep = joblib.load("models/preprocessing.joblib")

new_customer = pd.DataFrame(
    [{"age": 45, "annual_income": 98000, "monthly_spend": 1300, "visits_per_month": 12, "region": "metro"}]
)
new_X = pd.get_dummies(new_customer, dtype=float).reindex(columns=prep["feature_names"], fill_value=0.0)
probs = loaded.predict(prep["scaler"].transform(new_X.to_numpy()), verbose=0)[0]

for name, p in zip(prep["class_names"], probs):
    print(f"{name:10s} {p:.1%}")
print("Predicted segment:", prep["class_names"][int(np.argmax(probs))])